# A3 移植 (6/8)：改动后的一步前向与时序约束

> **一句话结论**：数据面骨架与 A5 相同，只是 resolver、失效、pack 换成 AICPU/AIV 实现；注意图捕获形状在 TP 对齐下被收敛，小批量会被 padding 到 16 重放 FULL 图。

## 1. 一步 decode 时序

1. 模型产出本步 KV / 查询；
2. **AICPU resolver**：翻译物理行 + 搬移/淘汰计划；
3. 缓存算子：gather 补缺、**AICPU invalidate** 作废旧索引；
4. **AIV 融合 pack**（decode 小批量）组装 656B 行；
5. SFA 在热区原地直读：int8 走 vendor 原生量化稀疏注意力；
6. 写回池并镜像 Host。

## 2. 图捕获形状与 padding 税

- 候选 capture sizes `[1, 2, 4, 8, 16]` 被 TP 对齐收敛为 **`[16]`**；
- 实际 bs1–15 会被 padding 到 16，重放 FULL 图；
- 慢点来自 15 个 dummy 请求稀释热区，**不是 eager 路径慢**；
- 因此主性能口径以 **bs16** 为准。

## 3. 算子捕获要求（A3 同样遵守）

- resolver：无堆分配，scratch 缓存复用；
- invalidate：捕获期不回退 Triton，录 AICPU task；
- pack：输出预分配，捕获路径无 host 同步；
- 相同输入产出稳定布局，保证图可重放。

## 4. 小结

- A3 前向骨架 = A5 骨架换引擎；
- 小批量 padding 到 16 重放 FULL 图，解读性能时注意口径；
- 下一篇读性能归因账本。

**下一篇**：`07-performance-attribution.ipynb`
